# Pré-processamento de dados — ParlaMint-PT

## 1. Configuração e carregamento

**Estrutura da Base de Dados** 

Temos duas pastas principais: 
- ParlaMint-PT.TEI e ParlaMint-PT.txt. Ambas contêm as mesmas informações, embora organizadas e estruturadas em diferentes formatos. Prosseguiremos com o .txt. 

In [1]:
import pandas as pd
from pathlib import Path

possible_data_dirs = [
    Path("../../ParlaMint-PT/ParlaMint-PT.txt"),
    Path("ParlaMint-PT/ParlaMint-PT.txt"),
]
data_dir = next((path for path in possible_data_dirs if path.is_dir()), None)
if data_dir is None:
    raise FileNotFoundError("Não foi encontrada a pasta ParlaMint-PT/ParlaMint-PT.txt.")
data_dir = data_dir.resolve()
project_dir = data_dir.parent.parent
output_dir = project_dir / "data"
output_dir.mkdir(exist_ok=True)

### Identificação dos ficheiros de metadados

In [2]:
tsv_files = sorted(data_dir.glob("*/*-meta.tsv"))
print(f"Found {len(tsv_files)} TSV files in {data_dir}")

Found 901 TSV files in /Users/alexandresousa/Desktop/Tese/ParlaMint-PT/ParlaMint-PT.txt


In [3]:
metadata_df = []

for file_path in tsv_files:
    session_metadata_df = pd.read_csv(
        file_path,
        sep="\t",
        dtype="string",
        na_values="-"
    )

    session_metadata_df["ficheiro_origem"] = str(file_path)
    metadata_df.append(session_metadata_df)

metadata = pd.concat(metadata_df, ignore_index=True)

metadata["Date"] = pd.to_datetime(metadata["Date"], errors="coerce")
metadata["ano"] = metadata["Date"].dt.year.astype("Int64")

print(f"Número de intervenções: {len(metadata):,}")
print(f"Número de sessões: {metadata['Text_ID'].nunique():,}")
      

KeyboardInterrupt: 

## 2. Verificação da estrutura e qualidade dos dados

In [ ]:
metadata.head()

                        Text_ID                            ID  \
0       ParlaMint-PT_2015-01-07    ParlaMint-PT_2015-01-07.u1   
1       ParlaMint-PT_2015-01-07    ParlaMint-PT_2015-01-07.u2   
2       ParlaMint-PT_2015-01-07    ParlaMint-PT_2015-01-07.u3   
3       ParlaMint-PT_2015-01-07    ParlaMint-PT_2015-01-07.u4   
4       ParlaMint-PT_2015-01-07    ParlaMint-PT_2015-01-07.u5   
...                         ...                           ...   
248572  ParlaMint-PT_2024-03-13  ParlaMint-PT_2024-03-13.u116   
248573  ParlaMint-PT_2024-03-13  ParlaMint-PT_2024-03-13.u117   
248574  ParlaMint-PT_2024-03-13  ParlaMint-PT_2024-03-13.u118   
248575  ParlaMint-PT_2024-03-13  ParlaMint-PT_2024-03-13.u119   
248576  ParlaMint-PT_2024-03-13  ParlaMint-PT_2024-03-13.u120   

                                                    Title        Date  \
0       Minutas da Assembleia da República Portuguesa,...  2015-01-07   
1       Minutas da Assembleia da República Portuguesa,...  2015-01-07   


In [ ]:
num_rows, num_cols = metadata.shape
print("Metadata Shape:", num_rows, "rows and", num_cols, "columns")

columns = metadata.columns
print("Columns:", columns)

#Tipo de colunas
column_types = metadata.dtypes
print("Column Types:", column_types)

Metadata Shape: 248577 rows and 26 columns
Columns: Index(['Text_ID', 'ID', 'Title', 'Date', 'Body', 'Term', 'Session', 'Meeting',
       'Sitting', 'Agenda', 'Subcorpus', 'Lang', 'Speaker_role', 'Speaker_MP',
       'Speaker_minister', 'Speaker_party', 'Speaker_party_name',
       'Party_status', 'Party_orientation', 'Speaker_ID', 'Speaker_name',
       'Speaker_gender', 'Speaker_birth', 'Topic', 'ficheiro_origem', 'ano'],
      dtype='object')
Column Types: Text_ID               string[python]
ID                    string[python]
Title                 string[python]
Date                  string[python]
Body                  string[python]
Term                  string[python]
Session               string[python]
Meeting               string[python]
Sitting               string[python]
Agenda                string[python]
Subcorpus             string[python]
Lang                  string[python]
Speaker_role          string[python]
Speaker_MP            string[python]
Speaker_minister  

Os campos estão todos em formato string. Vamos verificar se há dados em falta.

In [ ]:
#missing values
missing_values = metadata.isnull().sum()
print("Missing Values:\n", missing_values)

Missing Values:
 Text_ID                    0
ID                         0
Title                      0
Date                       0
Body                       0
Term                       0
Session                    0
Meeting                    0
Sitting                    0
Agenda                248577
Subcorpus                  0
Lang                       0
Speaker_role               0
Speaker_MP                 0
Speaker_minister           0
Speaker_party            262
Speaker_party_name       262
Party_status            1905
Party_orientation       1551
Speaker_ID                 0
Speaker_name               0
Speaker_gender             0
Speaker_birth           1743
Topic                      0
ficheiro_origem            0
ano                        0
dtype: int64


## 3. Revisão de colunas e tratamento de casos especiais

Verificar: Agenda, Party_status, Party_orientation, Speaker_party, Speaker_party_name e Speaker_birth.

In [ ]:
#Checking of the meaning of the columns that have missing values: Agenda, Speaker_Party, Speaker_party_name, Party_status, Party_orientation
agenda_column = metadata["Agenda"]

metadata = metadata.drop(columns="Agenda")

In [ ]:
linhas_sem_partido = metadata.loc[
    metadata[["Speaker_party", "Speaker_party_name"]].isna().any(axis=1)
].copy()

linhas_sem_partido["ficheiro_txt"] = (
    linhas_sem_partido["ficheiro_origem"]
    .str.replace("-meta.tsv", ".txt", regex=False)
)

ids_por_ficheiro = (
    linhas_sem_partido
    .groupby("ficheiro_txt")["ID"]
    .agg(set)
)

textos = []

for ficheiro_txt, ids in ids_por_ficheiro.items():
    texto_sessao = pd.read_csv(
        ficheiro_txt,
        sep="\t",
        header=None,
        names=["ID", "texto"],
        dtype="string"
    )

    textos.append(texto_sessao[texto_sessao["ID"].isin(ids)])

textos = pd.concat(textos, ignore_index=True)

linhas_sem_partido_com_texto = linhas_sem_partido.merge(
    textos,
    on="ID",
    how="left",
    validate="one_to_one"
)

display(linhas_sem_partido_com_texto)


,Text_ID,ID,Title,Date,Body,Term,Session,Meeting,Sitting,Subcorpus,...,Party_orientation,Speaker_ID,Speaker_name,Speaker_gender,Speaker_birth,Topic,ficheiro_origem,ano,ficheiro_txt,texto
0,ParlaMint-PT_2015-01-09,ParlaMint-PT_2015-01-09.u70,"Minutas da Assembleia da República Portuguesa,...",2015-01-09,Unicameralismo,Série I - XII Legislatura,Sessão 4,Número 36,09.01.2015,Referência,...,<NA>,AlexandreNunoVazBatistadeVieiraeBrito,"Nuno Vaz Batista de Vieira e Brito, Alexandre",M,<NA>,Agriculture,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,2015,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,"Sr.ª Presidente, Sr.as e Srs. Deputados: Retri..."
1,ParlaMint-PT_2015-01-09,ParlaMint-PT_2015-01-09.u72,"Minutas da Assembleia da República Portuguesa,...",2015-01-09,Unicameralismo,Série I - XII Legislatura,Sessão 4,Número 36,09.01.2015,Referência,...,<NA>,AlexandreNunoVazBatistadeVieiraeBrito,"Nuno Vaz Batista de Vieira e Brito, Alexandre",M,<NA>,Agriculture,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,2015,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,Os produtores deixaram de se rever no modelo v...
2,ParlaMint-PT_2015-01-09,ParlaMint-PT_2015-01-09.u74,"Minutas da Assembleia da República Portuguesa,...",2015-01-09,Unicameralismo,Série I - XII Legislatura,Sessão 4,Número 36,09.01.2015,Referência,...,<NA>,AlexandreNunoVazBatistadeVieiraeBrito,"Nuno Vaz Batista de Vieira e Brito, Alexandre",M,<NA>,Environment,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,2015,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,O Douro já responde assertivamente com duas as...
3,ParlaMint-PT_2015-01-09,ParlaMint-PT_2015-01-09.u90,"Minutas da Assembleia da República Portuguesa,...",2015-01-09,Unicameralismo,Série I - XII Legislatura,Sessão 4,Número 36,09.01.2015,Referência,...,<NA>,AlexandreNunoVazBatistadeVieiraeBrito,"Nuno Vaz Batista de Vieira e Brito, Alexandre",M,<NA>,Macroeconomics,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,2015,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,"Sr.ª Presidente, Srs. Deputados: Começo por ag..."
4,ParlaMint-PT_2015-01-15,ParlaMint-PT_2015-01-15.u2,"Minutas da Assembleia da República Portuguesa,...",2015-01-15,Unicameralismo,Série I - XII Legislatura,Sessão 4,Número 38,15.01.2015,Referência,...,<NA>,LeonardoBandeiradeMeloMathias,"Bandeira de Melo Mathias, Leonardo",M,<NA>,Domestic Commerce,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,2015,../ParlaMint-PT/ParlaMint-PT.txt/2015/ParlaMin...,"Sr.ª Presidente da Assembleia da República, Dr..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,ParlaMint-PT_2024-01-11,ParlaMint-PT_2024-01-11.u277,"Minutas da Assembleia da República Portuguesa,...",2024-01-11,Unicameralismo,Série I - XV Legislatura,Sessão 2,Número 39,11.01.2024,"COVID,Guerra",...,<NA>,PedroSousaRodrigues,"Sousa Rodrigues, Pedro",M,<NA>,Macroeconomics,../ParlaMint-PT/ParlaMint-PT.txt/2024/ParlaMin...,2024,../ParlaMint-PT/ParlaMint-PT.txt/2024/ParlaMin...,… e investimento no metro do Porto. Mas também...
258,ParlaMint-PT_2024-01-11,ParlaMint-PT_2024-01-11.u279,"Minutas da Assembleia da República Portuguesa,...",2024-01-11,Unicameralismo,Série I - XV Legislatura,Sessão 2,Número 39,11.01.2024,"COVID,Guerra",...,<NA>,PedroSousaRodrigues,"Sousa Rodrigues, Pedro",M,<NA>,Macroeconomics,../ParlaMint-PT/ParlaMint-PT.txt/2024/ParlaMin...,2024,../ParlaMint-PT/ParlaMint-PT.txt/2024/ParlaMin...,"São importantes para que o Governo possa, para..."
259,ParlaMint-PT_2024-01-11,ParlaMint-PT_2024-01-11.u513,"Minutas da Assembleia da República Portuguesa,...",2024-01-11,Unicameralismo,Série I - XV Legislatura,Sessão 2,Número 39,11.01.2024,"COVID,Guerra",...,<NA>,PedroSousaRodrigues,"Sousa Rodrigues, Pedro",M,<NA>,Macroeconomics,../ParlaMint-PT/ParlaMint-PT.txt/2024/ParlaMin...,2024,../ParlaMint-PT/ParlaMint-PT.txt/2024/ParlaMin...,"Sr. Presidente, Sr.as e Srs. Deputados: Assist..."
260,ParlaMint-PT_2024-01-11,ParlaMint-PT_2024-01-11.u515,"Minutas da Assemblei

In [ ]:
print((metadata["Speaker_role"] == "Convidado").sum())
metadata = metadata[metadata["Speaker_role"] != "Convidado"].copy()


4


Visualização dos valores das colunas uma a uma: 

In [ ]:
#Check different values in each column
for column in metadata.columns: 
    print(f"\n{column}:")
    print(metadata[column].unique())


Text_ID:
<StringArray>
['ParlaMint-PT_2015-01-07', 'ParlaMint-PT_2015-01-08',
 'ParlaMint-PT_2015-01-09', 'ParlaMint-PT_2015-01-14',
 'ParlaMint-PT_2015-01-15', 'ParlaMint-PT_2015-01-16',
 'ParlaMint-PT_2015-01-21', 'ParlaMint-PT_2015-01-22',
 'ParlaMint-PT_2015-01-28', 'ParlaMint-PT_2015-01-29',
 ...
 'ParlaMint-PT_2023-12-21', 'ParlaMint-PT_2024-01-03',
 'ParlaMint-PT_2024-01-04', 'ParlaMint-PT_2024-01-05',
 'ParlaMint-PT_2024-01-09', 'ParlaMint-PT_2024-01-10',
 'ParlaMint-PT_2024-01-11', 'ParlaMint-PT_2024-01-24',
 'ParlaMint-PT_2024-02-07', 'ParlaMint-PT_2024-03-13']
Length: 901, dtype: string

ID:
<StringArray>
[  'ParlaMint-PT_2015-01-07.u1',   'ParlaMint-PT_2015-01-07.u2',
   'ParlaMint-PT_2015-01-07.u3',   'ParlaMint-PT_2015-01-07.u4',
   'ParlaMint-PT_2015-01-07.u5',   'ParlaMint-PT_2015-01-07.u6',
   'ParlaMint-PT_2015-01-07.u7',   'ParlaMint-PT_2015-01-07.u8',
   'ParlaMint-PT_2015-01-07.u9',  'ParlaMint-PT_2015-01-07.u10',
 ...
 'ParlaMint-PT_2024-03-13.u111', 'ParlaMint-P

Através da visualização do número diferente de valores que uma coluna tem, podemos decidir que colunas remover pois não são precisas para esta análise em particular. Podemos também ver diferentes categorias das colunas que possam não ter necessárias ou que introduzam ruído

Podemos remove: Body, Lang (Tem só um valor) 

In [ ]:
metadata = metadata.drop(columns=["Body", "Lang"])

-----

4. Extrair os textos e dar Merge aos nossos metadados

In [ ]:
metadata = metadata.drop(
    columns=["texto", "ficheiro_txt"],
    errors = "ignore"
).copy()


metadata["ficheiro_txt"] = (
    metadata["ficheiro_origem"].str.replace("-meta.tsv", ".txt", regex = False)
)

ids_por_ficheiro = (
    metadata.groupby("ficheiro_txt")["ID"].agg(set)
)

textos = []

for ficheiro_txt, ids in ids_por_ficheiro.items():
    texto_sessao = pd.read_csv(
        ficheiro_txt,
        sep = "\t",
        header = None, 
        names = ["ID", "texto"], 
        dtype = "string"
    )
    
    textos.append(
        texto_sessao[texto_sessao["ID"].isin(ids)]
    )
    
textos = pd.concat(textos, ignore_index=True) 

if textos["ID"].duplicated().any():
    raise ValueError("Foram encontrados IDs duplicados nos ficheiros TXT.")

metadata = metadata.merge(
    textos, 
    on="ID",
    how = "left", 
    validate = "one_to_one"
)

if metadata["texto"].isna().any():
    raise ValueError(
        f"Há {metadata['texto'].isna().sum()} intervenções sem texto associado."
    )


Como já temos a coluna "Party_status" que nos diz o partido que faz parte do governo e os que fazem parte da oposição, vamos manter essa coluna e renomea-la para "bloco_politico". 

Assim: Coalition -> Governo e Opposition -> Oposição

In [ ]:
metadata = metadata.rename(
    columns={"Party_status": "bloco_politico"}
)

metadata["bloco_politico"] = metadata["bloco_politico"].replace({
    "Coalition": "Governo",
    "Opposition": "Oposição"
})

print(metadata["bloco_politico"].value_counts(dropna=False))

output_file = output_dir / "parlamint_pt_merged.csv"
metadata.to_csv(output_file, index=False, encoding="utf-8")
print(f"Tabela preparada em: {output_file}")

## 5. Exclusão de intervenções da presidência e da Mesa

Este bloco pode ser executado sem correr os anteriores: carrega `parlamint_pt_merged.csv` diretamente. Excluem-se os valores `Presidente de mesa` e `Mesa` de `Speaker_role`; outras intervenções procedimentais não identificadas por esta coluna são mantidas. Se o ficheiro já estiver filtrado, a contagem de novas exclusões será zero.

### Resultado histórico da primeira filtragem

Contagens registadas antes de substituir o CSV em 25 de outubro de 2026. O CSV atual já está filtrado; executar novamente o bloco seguinte conta apenas novas exclusões.

| Ano | Antes | Removidas | % removida |
| --- | ---: | ---: | ---: |
| 2015 | 22 601 | 6 115 | 27,1% |
| 2016 | 24 141 | 7 046 | 29,2% |
| 2017 | 25 949 | 7 659 | 29,5% |
| 2018 | 27 321 | 7 932 | 29,0% |
| 2019 | 26 358 | 7 535 | 28,6% |
| 2020 | 25 094 | 9 060 | 36,1% |
| 2021 | 18 821 | 6 987 | 37,1% |
| 2022 | 24 512 | 6 105 | 24,9% |
| 2023 | 50 648 | 9 877 | 19,5% |
| 2024 | 3 128 | 546 | 17,5% |
| **Total** | **248 573** | **68 862** | **27,7%** |

Restaram 179 711 intervenções. Estes valores são um registo histórico, não um cálculo efetuado a partir do CSV atual.

In [6]:
from pathlib import Path
import pandas as pd

possible_files = [
    Path("../../data/parlamint_pt_merged.csv"),
    Path("../data/parlamint_pt_merged.csv"),
    Path("data/parlamint_pt_merged.csv"),
]
output_file = next((path.resolve() for path in possible_files if path.is_file()), None)
if output_file is None:
    raise FileNotFoundError("Não foi encontrado data/parlamint_pt_merged.csv.")

metadata = pd.read_csv(output_file, dtype="string")
funcoes_mesa = {"presidente de mesa", "mesa"}
papel = metadata["Speaker_role"].str.strip().str.casefold()
mascara_mesa = papel.isin(funcoes_mesa)

intervencoes_antes = len(metadata)
intervencoes_removidas = int(mascara_mesa.sum())

relatorio_ano = pd.DataFrame({
    "antes": metadata.groupby("ano", dropna=False).size(),
    "removidas": metadata.loc[mascara_mesa].groupby("ano", dropna=False).size(),
}).fillna(0).astype(int)
relatorio_ano["restantes"] = relatorio_ano["antes"] - relatorio_ano["removidas"]
relatorio_ano["percentagem_removida"] = (
    100 * relatorio_ano["removidas"] / relatorio_ano["antes"]
).round(1)

metadata = metadata.loc[~mascara_mesa].copy()

print(f"Intervenções antes do filtro: {intervencoes_antes:,}")
print(f"Intervenções removidas: {intervencoes_removidas:,}")
if intervencoes_antes:
    print(f"Percentagem removida: {100 * intervencoes_removidas / intervencoes_antes:.1f}%")
print(f"Intervenções restantes: {len(metadata):,}")
if intervencoes_removidas == 0:
    print("Nenhuma nova intervenção da Mesa foi encontrada; o ficheiro pode já estar filtrado.")
display(relatorio_ano)

Intervenções antes do filtro: 179,711
Intervenções removidas: 0
Percentagem removida: 0.0%
Intervenções restantes: 179,711
Nenhuma nova intervenção da Mesa foi encontrada; o ficheiro pode já estar filtrado.


,antes,removidas,restantes,percentagem_removida
ano,,,,
2015,16486,0,16486,0.0
2016,17095,0,17095,0.0
2017,18290,0,18290,0.0
2018,19389,0,19389,0.0
2019,18823,0,18823,0.0
2020,16034,0,16034,0.0
2021,11834,0,11834,0.0
2022,18407,0,18407,0.0
2023,40771,0,40771,0.0


In [7]:
if intervencoes_removidas > 0:
    metadata.to_csv(output_file, index=False, encoding="utf-8")
    print(f"Tabela final guardada em: {output_file}")
else:
    print("O ficheiro não foi alterado: não há novas linhas a remover.")

O ficheiro não foi alterado: não há novas linhas a remover.
